# Build Mamba-SSM wheel for CUDA 12.2 + PTX

This notebook caches the source checkout in Google Drive, installs the build dependencies, and builds a wheel with PTX fallback enabled through `TORCH_CUDA_ARCH_LIST`.

It assumes a Colab GPU runtime with a CUDA 12.2-capable PyTorch install and builds a single `+PTX` wheel target. It does not install a prebuilt wheel.

In [ ]:
import os
import subprocess

REPO_URL = os.environ.get("REPO_URL", "https://github.com/davidkny22/efficient-mamba-ssm.git")
CACHE_DIR = "/content/drive/MyDrive/efficient_mamba_ssm"

if os.path.exists(CACHE_DIR):
    subprocess.run(["git", "-C", CACHE_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, CACHE_DIR], check=True)
os.chdir(CACHE_DIR)
print("repo cache:", os.getcwd())

In [ ]:
import os
import subprocess
import sys

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

print("Drive mounted; source cache lives at /content/drive/MyDrive/efficient_mamba_ssm")


In [ ]:
!python -m pip install --upgrade pip setuptools wheel ninja packaging
!python - <<'PY'
import torch
print('torch:', torch.__version__)
print('torch cuda:', torch.version.cuda)
PY

In [ ]:
import os

# Force a source build and add PTX fallback for CUDA 12.2.
os.environ["MAMBA_FORCE_BUILD"] = "TRUE"
os.environ["MAMBA_FORCE_CXX11_ABI"] = "FALSE"
os.environ["MAMBA_LOCAL_VERSION"] = "cu122ptx"
os.environ["MAX_JOBS"] = "2"
os.environ["TORCH_CUDA_ARCH_LIST"] = "12.2+PTX"

subprocess.run([sys.executable, "setup.py", "--name"], check=True)
subprocess.run([sys.executable, "setup.py", "bdist_wheel", "--dist-dir", "dist"], check=True)
subprocess.run(["bash", "-lc", "ls -lh dist"], check=True)


In [ ]:
import glob
print(glob.glob('dist/*.whl'))